In [1]:
from src.config import LLMParams

In [29]:
import asyncio
import json
import logging

import numpy as np
import random

import pandas as pd
from numpy.random import RandomState
from pydantic import TypeAdapter

from src.config import AppSettings, LLMConfig
from src.schemas import (
    PromptLogprob,
    PtbScenario,
    PtbScenarioRes,
    TokenEntropy,
    WordInfo,
    WordInfoRes,
    ReadingComprehensionItem,
    TokenEntropy,
)
from src.utils.base import configure_logging, create_openai_client, calculate_prompt_logprobs
from src.utils.tokens import calculate_token_entropy
from src.utils.words import get_words_and_indices

logger = logging.getLogger("research")

In [5]:
model = "Qwen/Qwen3-8B-FP8"
seed = 42
samples = 10

In [20]:
with open("../config_local.json", "r") as f:
    config = AppSettings.model_validate_json(f.read())

In [21]:
config.logging_conf_file='../logging_config.json'

In [22]:
configure_logging(config=config)

In [23]:
logger.info(config.model_dump_json(indent=1))

2026-02-14 10:59:34,266 ==INFO== research:<module>:1 ==--== {
 "logging_conf_file": "../logging_config.json",
 "llm": {
  "api_key": "**********",
  "base_url": "http://localhost:8996/v1",
  "timeout": 50,
  "async_cals": 5,
  "proxy_url": null,
  "params_extra": {
   "max_tokens": 5
  },
  "extra_body": {}
 },
 "entropy_threshold": 0.8,
 "max_entropy_scale": 2.5
}


In [24]:
df = pd.read_json("../MuSeRC/train.jsonl", lines=True)
small_df = df.sample(n=samples, random_state=RandomState(seed=seed))
del df

small_list = small_df.to_dict(orient="records")
ta = TypeAdapter(list[ReadingComprehensionItem])
items = ta.validate_python(small_list)

client = create_openai_client(config=config.llm)
semaphore = asyncio.Semaphore(config.llm.async_cals)

In [13]:
item = items[0]

random.seed(a=seed)
context = item.passage.text
question = random.choice(item.passage.questions)
answer = random.choice(question.answers)


scenario={
             "name": f"Запрос 0",
             "context": context,
             "reference": answer.text,
             "question": question.question,
         }

words_infos: list[WordInfo] = get_words_and_indices(scenario["context"])

In [14]:
words_infos

[{'word': '1', 'start': 1, 'end': 2},
 {'word': 'Спустя', 'start': 4, 'end': 10},
 {'word': 'какое', 'start': 11, 'end': 16},
 {'word': 'то', 'start': 17, 'end': 19},
 {'word': 'время', 'start': 20, 'end': 25},
 {'word': 'везение', 'start': 26, 'end': 33},
 {'word': 'прекращается', 'start': 34, 'end': 46},
 {'word': '2', 'start': 49, 'end': 50},
 {'word': 'Редакции', 'start': 52, 'end': 60},
 {'word': 'наперебой', 'start': 61, 'end': 70},
 {'word': 'стараются', 'start': 71, 'end': 80},
 {'word': 'обжулить', 'start': 81, 'end': 89},
 {'word': 'Мартина', 'start': 90, 'end': 97},
 {'word': '3', 'start': 100, 'end': 101},
 {'word': 'Добыть', 'start': 103, 'end': 109},
 {'word': 'у', 'start': 110, 'end': 111},
 {'word': 'них', 'start': 112, 'end': 115},
 {'word': 'деньги', 'start': 116, 'end': 122},
 {'word': 'за', 'start': 123, 'end': 125},
 {'word': 'публикации', 'start': 126, 'end': 136},
 {'word': 'оказывается', 'start': 137, 'end': 148},
 {'word': 'нелёгким', 'start': 149, 'end': 157},

In [15]:
scenario["context"]

'(1) Спустя какое-то время везение прекращается. (2) Редакции наперебой стараются обжулить Мартина. (3) Добыть у них деньги за публикации оказывается нелёгким делом. (4) Руфь настаивает на том, чтобы Мартин устроился на работу к её отцу, она не верит в то, что он станет писателем. (5) Случайно у Морзов Мартин знакомится с Рэссом Бриссенденом и близко сходится с ним. (6) Бриссенден болен чахоткой, он не боится смерти, но страстно любит жизнь во всех её проявлениях. (7) Бриссенден знакомит Мартина с «настоящими людьми», одержимыми литературой и философией. (8) Со своим новым товарищем Мартин посещает митинг социалистов, где спорит с оратором, но благодаря расторопному и нещепетильному репортёру попадает на страницы газет в качестве социалиста и ниспровергателя существующего строя. (9) Газетная публикация приводит к печальным последствиям — Руфь присылает Мартину письмо, извещающее о разрыве помолвки. (10) Мартин продолжает жить по инерции, и его даже не радуют поступающие от журналов чек

In [16]:
text = "context: " + scenario["context"] + "\nquestion: " + scenario["question"]

In [17]:
text

'context: (1) Спустя какое-то время везение прекращается. (2) Редакции наперебой стараются обжулить Мартина. (3) Добыть у них деньги за публикации оказывается нелёгким делом. (4) Руфь настаивает на том, чтобы Мартин устроился на работу к её отцу, она не верит в то, что он станет писателем. (5) Случайно у Морзов Мартин знакомится с Рэссом Бриссенденом и близко сходится с ним. (6) Бриссенден болен чахоткой, он не боится смерти, но страстно любит жизнь во всех её проявлениях. (7) Бриссенден знакомит Мартина с «настоящими людьми», одержимыми литературой и философией. (8) Со своим новым товарищем Мартин посещает митинг социалистов, где спорит с оратором, но благодаря расторопному и нещепетильному репортёру попадает на страницы газет в качестве социалиста и ниспровергателя существующего строя. (9) Газетная публикация приводит к печальным последствиям — Руфь присылает Мартину письмо, извещающее о разрыве помолвки. (10) Мартин продолжает жить по инерции, и его даже не радуют поступающие от жур

In [25]:
answer, prompt_l = await calculate_prompt_logprobs(
    idx=str(item.idx),
    query=text,
    client=client,
    semaphore=semaphore,
    model=model,
    config=config.llm,
)

2026-02-14 10:59:46,703 ==DEBUG== src.utils.base:calculate_prompt_logprobs:72 ==--== Start request id 361


In [26]:
prompt_l

[None,
 {'872': PromptLogprob(decoded_token='user', logprob=-10.38395881652832, rank=4056),
  '67': PromptLogprob(decoded_token='d', logprob=-7.1964592933654785, rank=1),
  '82': PromptLogprob(decoded_token='s', logprob=-7.5714592933654785, rank=2),
  '258': PromptLogprob(decoded_token='in', logprob=-7.6027092933654785, rank=3),
  '15136': PromptLogprob(decoded_token='times', logprob=-7.6027092933654785, rank=4),
  '8': PromptLogprob(decoded_token=')', logprob=-7.6652092933654785, rank=5)},
 {'198': PromptLogprob(decoded_token='\n', logprob=-0.14488349854946136, rank=1),
  '271': PromptLogprob(decoded_token='\n\n', logprob=-2.269883394241333, rank=2),
  '25': PromptLogprob(decoded_token=':', logprob=-4.144883632659912, rank=3),
  '2610': PromptLogprob(decoded_token='You', logprob=-5.144883632659912, rank=4),
  '40': PromptLogprob(decoded_token='I', logprob=-6.644883632659912, rank=5)},
 {'2147': PromptLogprob(decoded_token='context', logprob=-23.620494842529297, rank=68146),
  '2610': 

In [30]:
data: list[TokenEntropy] = []
for forward in prompt_l:
    if not isinstance(forward, dict):
        continue
    logprobs: list[PromptLogprob] = []
    token_str = None
    for _, logprob in forward.items():
        logprobs.append(logprob)
        if not isinstance(token_str, str):
            token_str = logprob.decoded_token

    entropy = calculate_token_entropy(logprobs)
    data.append({"token": str(token_str), "entropy": entropy})


In [31]:
data

[{'token': 'user', 'entropy': np.float64(2.3612452025168307)},
 {'token': '\n', 'entropy': np.float64(0.6630443670161185)},
 {'token': 'context', 'entropy': np.float64(0.7592331329468887)},
 {'token': ':', 'entropy': np.float64(0.7708924274654123)},
 {'token': ' (', 'entropy': np.float64(1.9926661336370701)},
 {'token': '1', 'entropy': np.float64(2.21897628729133)},
 {'token': ')', 'entropy': np.float64(0.18189695172339873)},
 {'token': ' С', 'entropy': np.float64(2.1471855530649337)},
 {'token': 'пуст', 'entropy': np.float64(1.8531446434887968)},
 {'token': 'я', 'entropy': np.float64(0.010369090646784093)},
 {'token': ' как', 'entropy': np.float64(1.9422618945661823)},
 {'token': 'ое', 'entropy': np.float64(0.03771490476598466)},
 {'token': '-', 'entropy': np.float64(1.00366648358768)},
 {'token': 'то', 'entropy': np.float64(0.0014393621278972236)},
 {'token': ' время', 'entropy': np.float64(0.003654559004685574)},
 {'token': ' в', 'entropy': np.float64(1.921951390081493)},
 {'token':

In [43]:
entropy2token: list[TokenEntropy] = []
prompt_buffer: str = ""  # буфер текста
prompt_tokens_map: list[int] = []  # мапинг текста на id токена

counter = 0
for forward in prompt_l:
    if not isinstance(forward, dict):
        continue
    logprobs: list[PromptLogprob] = []
    token_str = None
    for _, logprob in forward.items():
        logprobs.append(logprob)
        if not isinstance(token_str, str):
            token_str = logprob.decoded_token

    entropy = calculate_token_entropy(logprobs)
    entropy2token.append({"token": str(token_str), "entropy": entropy})

    prompt_buffer = prompt_buffer + str(token_str)
    prompt_tokens_map.extend([counter] * len(str(token_str)))
    counter += 1


In [35]:
prompt_buffer

'user\ncontext: (1) Спустя какое-то время везение прекращается. (2) Редакции наперебой стараются обжулить Мартина. (3) Добыть у них деньги за публикации оказывается нелёгким делом. (4) Руфь настаивает на том, чтобы Мартин устроился на работу к её отцу, она не верит в то, что он станет писателем. (5) Случайно у Морзов Мартин знакомится с Рэссом Бриссенденом и близко сходится с ним. (6) Бриссенден болен чахоткой, он не боится смерти, но страстно любит жизнь во всех её проявлениях. (7) Бриссенден знакомит Мартина с «настоящими людьми», одержимыми литературой и философией. (8) Со своим новым товарищем Мартин посещает митинг социалистов, где спорит с оратором, но благодаря расторопному и нещепетильному репортёру попадает на страницы газет в качестве социалиста и ниспровергателя существующего строя. (9) Газетная публикация приводит к печальным последствиям — Руфь присылает Мартину письмо, извещающее о разрыве помолвки. (10) Мартин продолжает жить по инерции, и его даже не радуют поступающие 

In [36]:
# INFO: обрезаем всё кроме контекста
start_i = prompt_buffer.find("context: ") + 9
end_i = prompt_buffer.find("question: ")


In [37]:
prompt_buffer[start_i:end_i]

'(1) Спустя какое-то время везение прекращается. (2) Редакции наперебой стараются обжулить Мартина. (3) Добыть у них деньги за публикации оказывается нелёгким делом. (4) Руфь настаивает на том, чтобы Мартин устроился на работу к её отцу, она не верит в то, что он станет писателем. (5) Случайно у Морзов Мартин знакомится с Рэссом Бриссенденом и близко сходится с ним. (6) Бриссенден болен чахоткой, он не боится смерти, но страстно любит жизнь во всех её проявлениях. (7) Бриссенден знакомит Мартина с «настоящими людьми», одержимыми литературой и философией. (8) Со своим новым товарищем Мартин посещает митинг социалистов, где спорит с оратором, но благодаря расторопному и нещепетильному репортёру попадает на страницы газет в качестве социалиста и ниспровергателя существующего строя. (9) Газетная публикация приводит к печальным последствиям — Руфь присылает Мартину письмо, извещающее о разрыве помолвки. (10) Мартин продолжает жить по инерции, и его даже не радуют поступающие от журналов чек

In [44]:
for token_id, simbol in zip(prompt_tokens_map, prompt_buffer):
    enthropy_info = entropy2token[token_id]
    print(f"ID: {token_id}; '{simbol}'; {enthropy_info}")

ID: 0; 'u'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 0; 's'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 0; 'e'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 0; 'r'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 1; '
'; {'token': '\n', 'entropy': np.float64(0.6630443670161185)}
ID: 2; 'c'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'o'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'n'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 't'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'e'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'x'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 't'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 3; ':'; {'token': ':', 'entropy': np.float64(0.7708924274654123)}
ID: 4; ' '; {'token': ' (', 'entrop

In [44]:
for token_id, simbol in zip(prompt_tokens_map, prompt_buffer):
    enthropy_info = entropy2token[token_id]
    print(f"ID: {token_id}; '{simbol}'; {enthropy_info}")

ID: 0; 'u'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 0; 's'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 0; 'e'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 0; 'r'; {'token': 'user', 'entropy': np.float64(2.3612452025168307)}
ID: 1; '
'; {'token': '\n', 'entropy': np.float64(0.6630443670161185)}
ID: 2; 'c'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'o'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'n'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 't'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'e'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 'x'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 2; 't'; {'token': 'context', 'entropy': np.float64(0.7592331329468887)}
ID: 3; ':'; {'token': ':', 'entropy': np.float64(0.7708924274654123)}
ID: 4; ' '; {'token': ' (', 'entrop

In [56]:
prompt_buffer_c = prompt_buffer[start_i:end_i]
prompt_tokens_map_c = prompt_tokens_map[start_i:end_i]
res_words: list[WordInfoRes] = []

current_pos = 0
for word in words_infos:
    try:
        start_idx = prompt_buffer_c.index(word["word"], current_pos)
    except ValueError:
        logger.warning(
            f"Слово '{word}' не найдено в тексте токенов начиная с позиции {current_pos}."
        )
        res_words.append(WordInfoRes(entropy=0.0, **word))
        continue

    end_idx = start_idx + len(word["word"])

    # Собираем все уникальные токены, которые попали в диапазон слова
    # Используем set для уникальности, затем сортируем
    matched_token_indices = sorted(list(set(prompt_tokens_map_c[start_idx:end_idx])))

    # вычисляем энтропию и нормализуем
    # word_entropy = float(
    #     sum([entropy2token[i]["entropy"] for i in matched_token_indices])
    # )
    # n_word_entropy = word_entropy / len(matched_token_indices)
    match_i = [i for i in matched_token_indices]
    idx = match_i[0]
    n_word_entropy = float(
        entropy2token[idx]['entropy']
    )

    res_words.append(WordInfoRes(entropy=n_word_entropy, **word))


In [57]:
print(*res_words,sep="\n")

{'entropy': 2.21897628729133, 'word': '1', 'start': 1, 'end': 2}
{'entropy': 2.1471855530649337, 'word': 'Спустя', 'start': 4, 'end': 10}
{'entropy': 1.9422618945661823, 'word': 'какое', 'start': 11, 'end': 16}
{'entropy': 0.0014393621278972236, 'word': 'то', 'start': 17, 'end': 19}
{'entropy': 0.003654559004685574, 'word': 'время', 'start': 20, 'end': 25}
{'entropy': 1.921951390081493, 'word': 'везение', 'start': 26, 'end': 33}
{'entropy': 2.383061382969207, 'word': 'прекращается', 'start': 34, 'end': 46}
{'entropy': 0.014787835713112398, 'word': '2', 'start': 49, 'end': 50}
{'entropy': 2.232938633591899, 'word': 'Редакции', 'start': 52, 'end': 60}
{'entropy': 2.2445767519432884, 'word': 'наперебой', 'start': 61, 'end': 70}
{'entropy': 2.14747147936252, 'word': 'стараются', 'start': 71, 'end': 80}
{'entropy': 2.4402020575874985, 'word': 'обжулить', 'start': 81, 'end': 89}
{'entropy': 2.21578386673578, 'word': 'Мартина', 'start': 90, 'end': 97}
{'entropy': 0.0028991607296633163, 'word'

In [ ]:
text = "context: " + scenario["context"] + "\nquestion: " + scenario["question"]

_, tokens_result = analyze_prompt_entropy(
    idx=str(item.idx),
    scenario={"name": f"Запрос 0", "text": text},
    client=client,
    semaphore=semaphore,
    model=model,
    config=config.llm,
)